# Build vs. Buy — Economic Model

**The question.** For a new AI feature, do we self-host an open-source model or call a SaaS API? The honest answer is: it depends on the interaction of three variables — request volume, unit cost, and engineering time — across time.

**What this notebook produces.**
1. A parameterized 24-month total-cost-of-ownership (TCO) model for both paths.
2. A break-even curve: at what request volume does self-hosting win?
3. A sensitivity analysis on the three most fragile assumptions.
4. A decision rule expressed in the form a PM would actually use in a room.

**Why this matters.** Build-vs-buy is usually decided on vibes. A quantified model is boring, survives executive scrutiny, and — most importantly — reveals when the decision is close enough that the operational factors (team skill, time-to-market, vendor lock-in) should break the tie, not the dollars.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Parameterize the scenario

Inputs below are anchored to realistic mid-2020s numbers for a medium-volume B2C AI feature. Change them to match a specific product.

In [ ]:
@dataclass
class BuildScenario:
    """Self-host open-source model on cloud infra."""
    monthly_infra_usd: float = 4_200       # GPU VM, storage, egress
    one_time_engineering_usd: float = 75_000  # initial productionization
    monthly_maintenance_hours: float = 20  # ongoing ops
    blended_engineer_hourly_usd: float = 150
    cost_per_request_usd: float = 0.0      # marginal cost is ~0 at steady state

@dataclass
class BuyScenario:
    """Pay a SaaS API per request."""
    cost_per_request_usd: float = 0.012    # ~$12 per 1k calls
    monthly_fixed_usd: float = 200         # minimum commit
    one_time_integration_usd: float = 8_000
    monthly_maintenance_hours: float = 3
    blended_engineer_hourly_usd: float = 150

build = BuildScenario()
buy = BuyScenario()

HORIZON_MONTHS = 24
months = np.arange(1, HORIZON_MONTHS + 1)

## 2. TCO functions

In [ ]:
def tco_build(monthly_requests: float, months: np.ndarray, s: BuildScenario) -> np.ndarray:
    ongoing_monthly = (
        s.monthly_infra_usd
        + s.monthly_maintenance_hours * s.blended_engineer_hourly_usd
        + monthly_requests * s.cost_per_request_usd
    )
    cumulative = s.one_time_engineering_usd + np.cumsum(np.full_like(months, ongoing_monthly, dtype=float))
    return cumulative

def tco_buy(monthly_requests: float, months: np.ndarray, s: BuyScenario) -> np.ndarray:
    ongoing_monthly = (
        s.monthly_fixed_usd
        + s.monthly_maintenance_hours * s.blended_engineer_hourly_usd
        + monthly_requests * s.cost_per_request_usd
    )
    cumulative = s.one_time_integration_usd + np.cumsum(np.full_like(months, ongoing_monthly, dtype=float))
    return cumulative

## 3. Compare at three volume tiers

Plot cumulative cost over 24 months at low / medium / high volume.

In [ ]:
volume_tiers = {
    "Low (100k/mo)": 100_000,
    "Medium (1M/mo)": 1_000_000,
    "High (10M/mo)": 10_000_000,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
for ax, (label, vol) in zip(axes, volume_tiers.items()):
    b = tco_build(vol, months, build)
    p = tco_buy(vol, months, buy)
    ax.plot(months, b / 1_000, label="Build (self-host)", color="#6f42c1", linewidth=2)
    ax.plot(months, p / 1_000, label="Buy (SaaS API)", color="#2ea043", linewidth=2)
    ax.set_title(label)
    ax.set_xlabel("Month")
    ax.set_ylabel("Cumulative cost ($k)")
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Break-even volume

At what monthly volume does build's 24-month TCO equal buy's? Below this volume, buy wins; above, build wins.

In [ ]:
volumes = np.logspace(4, 7.5, 60)
build_tco_24 = np.array([tco_build(v, months, build)[-1] for v in volumes])
buy_tco_24 = np.array([tco_buy(v, months, buy)[-1] for v in volumes])

crossover_idx = np.argmin(np.abs(build_tco_24 - buy_tco_24))
crossover_volume = volumes[crossover_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(volumes, build_tco_24 / 1_000, label="Build 24-mo TCO", color="#6f42c1", linewidth=2)
ax.plot(volumes, buy_tco_24 / 1_000, label="Buy 24-mo TCO", color="#2ea043", linewidth=2)
ax.axvline(crossover_volume, color="#bf3989", linestyle="--", alpha=0.7,
           label=f"Break-even: {crossover_volume:,.0f} req/mo")
ax.set_xscale("log")
ax.set_xlabel("Monthly requests")
ax.set_ylabel("24-month cumulative cost ($k)")
ax.set_title("Build vs. Buy — break-even at 24 months")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Break-even monthly volume (24-mo horizon): ~{crossover_volume:,.0f} requests")

## 5. Sensitivity analysis

The break-even volume is most fragile to three assumptions: SaaS per-request price, self-host monthly infra cost, and build engineering hours. Sweep ±50% on each and show the effect.

In [ ]:
def break_even_volume(build_s: BuildScenario, buy_s: BuyScenario) -> float:
    vols = np.logspace(4, 7.5, 200)
    deltas = np.array([tco_build(v, months, build_s)[-1] - tco_buy(v, months, buy_s)[-1] for v in vols])
    idx = np.argmin(np.abs(deltas))
    return float(vols[idx])

multipliers = [0.5, 0.75, 1.0, 1.25, 1.5]
sensitivities = {"buy.cost_per_request_usd": [], "build.monthly_infra_usd": [], "build.one_time_engineering_usd": []}

for m in multipliers:
    b1 = BuildScenario(**{**build.__dict__})
    p1 = BuyScenario(**{**buy.__dict__, "cost_per_request_usd": buy.cost_per_request_usd * m})
    sensitivities["buy.cost_per_request_usd"].append(break_even_volume(b1, p1))

    b2 = BuildScenario(**{**build.__dict__, "monthly_infra_usd": build.monthly_infra_usd * m})
    p2 = BuyScenario(**{**buy.__dict__})
    sensitivities["build.monthly_infra_usd"].append(break_even_volume(b2, p2))

    b3 = BuildScenario(**{**build.__dict__, "one_time_engineering_usd": build.one_time_engineering_usd * m})
    p3 = BuyScenario(**{**buy.__dict__})
    sensitivities["build.one_time_engineering_usd"].append(break_even_volume(b3, p3))

sens_df = pd.DataFrame(sensitivities, index=[f"{int(m*100)}%" for m in multipliers])
print("Break-even volume (req/mo) at ±50% variable multiplier:")
print(sens_df.round(-3).astype(int).to_string())

## Decision rule for a PM conversation

> **Buy if:** monthly volume is below ~break-even AND time-to-market < 2 months AND the feature is still validating product-market fit.
>
> **Build if:** monthly volume is above ~break-even AND the capability is strategic AND the team has the ML ops muscle to own it.
>
> **Close call:** if within ±25% of break-even, the dollars are not the deciding factor — vendor lock-in risk, data-residency, model-drift ownership, and team skill should break the tie.

The real value of this model isn't the point estimate — it's that the sensitivity table tells the PM which assumption to stress-test first before the executive review. In practice that was SaaS per-request price (vendors raise prices) and build maintenance hours (always undercounted).